In [11]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from env.charging_env import ChargingStationEnv
from metrics.validate import report_queueing_laws, validate_episode

from policy.power.proportional import ProportionalPower
from policy.power.static import StaticPower
from policy.queue.fifo import FIFOQueuePolicy
from policy.queue.lowest_soc_diff import LowestSoCDiffQueuePolicy
from visualization.ev_curves import plot_ev_theory_vs_sim
from visualization.pile_power import plot_pile_connector_power
from config import MAX_TIME
from simulation.arrivals import generate_arrivals

from __future__ import annotations

from offline_cl_opt import (
    StationSpec,
    build_cl_model,
    solve_cl_model,
    vehicles_from_evs,
)
from offline_cl_opt.solution import extract_solution


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
# Shared station / traffic inputs (simulation and offline use the same layout).
SCENARIO = {
    "n_piles": 1, # Typical 2 - 4
    "n_connectors": 2, # Typical 2
    "n_modules": 6, # 4 - 8
    "p_module": 25.0, # 25 - 50 kW; pile power 150 - 400 kW
    "queue_capacity": 1000, #
}

# Simulation controls
HR2MIN = 60
MAX_TIME = HR2MIN * 2   # length of the MEASURED phase (minutes) -- see WARMUP_PERIOD below
MEAN_INTERARRIVAL = 30.0
DELTA = 1.0
# None: keep continuous exponential times.
# 1: nearest minute; 2/3/5: nearest multiple of that many minutes.
DELTA_ARR = DELTA
BATTERY_CAP_OPTIONS = [75] # Typical 75, test [50, 75, 100]
BATTERY_CAP_OPTIONS = [cap * HR2MIN for cap in BATTERY_CAP_OPTIONS]
SEED = 39
POLICY_SEED = 12
QUEUE_POLICY = FIFOQueuePolicy()
POWER_POLICY = ProportionalPower()

# --- Warm-up period -- usage example ------------------------------------
# See simulation/engine.py's own module docstring ("Warm-up period") and
# metrics/metrics_tracker.py's for the full mechanics. Short version: the
# DES runs WARMUP_PERIOD minutes first, then MAX_TIME more (the "measured"
# phase); queue/pile state at the boundary carries straight over (nobody is
# reset), so the measured phase doesn't start from an empty station -- lets
# transient start-up effects (empty queue, all piles idle) settle out
# before the window you actually report metrics over.
#
# ChargingStationEnv(..., warmup_period=None) -- the default -- reads
# config.WARMUP_PERIOD (0.0 out of the box, i.e. no warm-up at all,
# fully backward compatible). Pass an explicit value, as below, to
# override it for this run without touching config.py.
WARMUP_PERIOD = HR2MIN * 6  # here: a long warm-up relative to MAX_TIME, deliberately, to reach steady state

# If True, empty the live QUEUE only (never vehicles already charging --
# see metrics.in_service_at_warmup_end below) at the exact instant warm-up
# ends, so the measured window's own queueing dynamics don't inherit
# whatever backlog built up during warm-up. Off by default: usually you
# want the measured phase to see a realistic, already-loaded system, not
# an artificially emptied queue.
FLUSH_QUEUE_AT_WARMUP = False

# 1. Generate the EV list yourself (same object format the simulation uses internally)
# max_time here must cover the WHOLE run (warm-up + measured) -- oversample
# generously since this is a one-shot draw, not resampled per phase.
rng_arrival = np.random.default_rng(SEED)
evs = generate_arrivals(
    mean_interarrival=MEAN_INTERARRIVAL,
    max_time=(WARMUP_PERIOD + MAX_TIME) * 1.5, rng=rng_arrival,
    battery_cap_options=BATTERY_CAP_OPTIONS,
    delta_arr=DELTA_ARR,
    )


In [13]:
#Feed it to the simulation: mean_interarrival=None so the external list is used
env = ChargingStationEnv(
    **SCENARIO,
    power_policy=POWER_POLICY,
    mean_interarrival=None,   # None -> use `arrivals`, not internal sampling
    arrivals=evs,
    max_time=MAX_TIME,
    battery_cap_options=BATTERY_CAP_OPTIONS,
    delta_arr=DELTA_ARR,
    warmup_period=WARMUP_PERIOD,              # usage example -- see the cell above
    flush_queue_at_warmup=FLUSH_QUEUE_AT_WARMUP,
)
obs, info = env.reset(seed=SEED)

done = False
rng = np.random.default_rng(POLICY_SEED)
policy = QUEUE_POLICY
total_reward = 0.0

t0 = time.perf_counter()
while not done:
    mask = env.action_masks()
    ev, pile_id = policy.decide(obs, mask, rng, env.engine.station)
    obs, reward, done, _, info = env.step(pile_id, ev=ev)
    total_reward += reward
sim_runtime = time.perf_counter() - t0

m = env.engine.metrics
finished = m.finished_evs
arrived = m.arrived_evs
dropped = m.dropped_evs

# Kept as whole-run scalars (downstream cells reference these by name) --
# same values as the "whole-run" side of the two-window print below.
sim_total_sojourn = float(sum(ev.departure_time - ev.arrival_time for ev in finished))
sim_mean_sojourn = m.mean_sojourn()

print(f"Finished EVs: {len(finished)}")
print(f"Arrived EVs:  {len(arrived)}")
print(f"Dropped EVs:  {len(dropped)}")
print(f"Total reward: {total_reward:.3f}")
print(f"Sim time (episode clock, TOTAL incl. warm-up): {env.engine.current_time:.1f} min "
      f"(warm-up={env.engine.warmup_period:.1f} + measured={env.engine.measurement_horizon:.1f})")
print(f"Sim wall-clock runtime:   {sim_runtime:.3f} s")

# --- Metrics, two windows -------------------------------------------------
# Everything below is reported BOTH ways: over the whole simulated run
# (including warm-up) and restricted to the measured phase only
# (post_warmup_only=True / since=m.warmup_period). See MetricsTracker's own
# module docstring, "Warm-up period", for exactly which quantities are
# time-windowed (L, Q, utilization, energy) vs. cohort-filtered (per-EV
# wait/service/sojourn -- excludes anyone who *arrived* during warm-up,
# even if they finish after it) by `since`.
print("\n=== Sojourn: whole run vs. measured-phase-only ===")
print(f"  Sim total sojourn  whole-run={sim_total_sojourn:.2f} min   "
      f"measured-only={m.finished_time_arrays(m.warmup_period)[2].sum():.2f} min")
print(f"  Sim mean  sojourn  whole-run={sim_mean_sojourn:.2f} min   "
      f"measured-only={m.mean_sojourn(m.warmup_period):.2f} min")
print(f"  Avg L  whole-run={m.average_L():.3f}   measured-only={m.average_L(m.warmup_period):.3f}")
print(f"  Avg Q  whole-run={m.average_Q():.3f}   measured-only={m.average_Q(m.warmup_period):.3f}")

# validate_episode's conservation check accounts for the warm-up flush on
# its own (arrived = finished + dropped + flushed + queued + plugged); it
# is always a whole-run check (see its own docstring -- it's a live-state
# audit, not something that has a "since" version).
rep_val = validate_episode(env, verbose=True)

# report_queueing_laws(post_warmup_only=...) is the single switch for the
# Little's-law summary: False (default) = whole run; True = measured phase
# only. Both calls below share the exact same MetricsTracker.queueing_summary
# under the hood -- see metrics/validate.py.
print("\n--- Whole run ---")
report_full = report_queueing_laws(env, verbose=True, post_warmup_only=False)
print("\n--- Measured phase only (post-warm-up) ---")
report_post = report_queueing_laws(env, verbose=True, post_warmup_only=True)

# --- Boundary-condition lists (for a future optimization run confined to
# just the measured window -- see MetricsTracker's own module docstring) --
print(f"\nqueued_at_warmup_end:     {len(m.queued_at_warmup_end)} vehicle(s) waiting when the measured phase began")
print(f"in_service_at_warmup_end: {len(m.in_service_at_warmup_end)} vehicle(s) already mid-charge when the measured phase began")
print(f"arrived_post_warmup:      {len(m.arrived_post_warmup)} vehicle(s) that arrived during the measured phase")
print(f"flushed_evs:              {len(m.flushed_evs)} vehicle(s) discarded from the queue at the boundary "
      f"(only nonempty if FLUSH_QUEUE_AT_WARMUP=True)")


Finished EVs: 17
Arrived EVs:  17
Dropped EVs:  0
Total reward: -69.038
Sim time (episode clock, TOTAL incl. warm-up): 480.0 min (warm-up=360.0 + measured=120.0)
Sim wall-clock runtime:   0.018 s

=== Sojourn: whole run vs. measured-phase-only ===
  Sim total sojourn  whole-run=517.78 min   measured-only=252.47 min
  Sim mean  sojourn  whole-run=30.46 min   measured-only=42.08 min
  Avg L  whole-run=1.079   measured-only=2.215
  Avg Q  whole-run=0.144   measured-only=0.606

=== Episode validation ===
  arrived=17, finished=17, dropped=0, flushed=0, queued=0, plugged=0
  [PASS] Finished: s_current ~= s_f: ok for all finished EVs
  [PASS] Finished: energy_received ~= c_b*(s_f-s_i)/HR2MIN: ok for all finished EVs
  [PASS] Finished: energy_received ~= energy_received_2: ok for all finished EVs
  [PASS] Finished: departure_time >= service_start_time >= arrival_time: ok for all finished EVs
  [PASS] Finished: pile_tracker / connector_id_tracker set: ok for all finished EVs
  [PASS] Finished:

In [14]:
# =============================================================================
# Measured-phase reporting: WHICH vehicles count?  (cohort flags)
# =============================================================================
# With a warm-up, three disjoint groups of vehicles are present in the
# measured window, and every reported number depends on which of them you
# count. They mirror offline_cl_opt.boundary.Cohort exactly, so the
# simulation and the optimization can be made to report the same set:
#
#   measurement -- arrived AFTER the boundary. This is what the plain
#                  `since=warmup_period` filter gives you.
#   queued      -- were WAITING in the queue at the boundary
#                  (metrics.queued_at_warmup_end)
#   boundary    -- were ALREADY PLUGGED IN at the boundary
#                  (metrics.in_service_at_warmup_end)
#
# The last two arrived *before* warmup_period, so `since` drops them. Add
# them back by id with `include_ids=m.cohort_ev_ids(...)`.
print(f"cohort sizes:  queued={len(m.queued_at_warmup_end)}   "
      f"boundary={len(m.in_service_at_warmup_end)}   "
      f"measurement={len(m.arrived_post_warmup)}")

# --- The three nested reports, in one call --------------------------------
# truncate_included=True (the default) clocks the added vehicles from the
# BOUNDARY rather than from their real arrival, i.e. counts only the part of
# their stay inside the measured window. That is what makes these numbers
# comparable with the offline models, which place the same vehicles at a=0.
print("\n=== Simulation sojourn by cohort (truncated at the boundary) ===")
for name, s in m.sojourn_by_cohort(since=m.warmup_period).items():
    print(f"  {name:20s} n={int(s['n']):4d}  total={s['total_sojourn']:9.2f} min"
          f"  mean={s['mean_sojourn']:7.2f} min")

# truncate_included=False gives their TRUE end-to-end times instead (arrival
# to departure, including the part that happened during warm-up). Right for
# "how long did this driver actually wait", but NOT comparable against an
# optimizer's mean sojourn -- the optimizer never sees the pre-boundary part.
print("\n=== Same, but true end-to-end times (NOT optimizer-comparable) ===")
for name, s in m.sojourn_by_cohort(since=m.warmup_period, truncate_included=False).items():
    print(f"  {name:20s} n={int(s['n']):4d}  total={s['total_sojourn']:9.2f} min"
          f"  mean={s['mean_sojourn']:7.2f} min")

# --- The same flags on the individual metrics ------------------------------
# Every per-customer metric takes the same include_ids/truncate_included
# keywords: mean_sojourn, mean_wait, max_wait, mean_service,
# service_squared_cv, and queueing_summary.
both = m.cohort_ev_ids(queued=True, boundary=True)
print()
print(f"mean_sojourn   whole run                    = {m.mean_sojourn():.2f} min")
print(f"mean_sojourn   measured phase only           = {m.mean_sojourn(m.warmup_period):.2f} min")
print(f"mean_sojourn   measured + queued + boundary  = "
      f"{m.mean_sojourn(m.warmup_period, include_ids=both):.2f} min")

# queueing_summary too. NOTE: the flags change only the per-CUSTOMER averages
# (W, W_q, S, n_finished and the lambda_eff/L_theory/Q_theory built from
# them). L_sim/Q_sim/rho_sim come from the online occupancy history, which
# already counts every vehicle physically present, whatever cohort it is in.
qs = m.queueing_summary(env.engine.current_time, since=m.warmup_period, include_ids=both)
print(f"queueing_summary(+cohorts): n_finished={int(qs['n_finished'])}  "
      f"W={qs['W']:.2f}  W_q={qs['W_q']:.2f}  L_sim={qs['L_sim']:.3f}")

# CAVEAT worth remembering when comparing against the optimizer: the
# simulation averages only vehicles that FINISHED inside the run, while the
# offline model reports every vehicle it was given (censoring unfinished
# ones at the horizon, D_j = K). With a short measured phase those are
# different denominators -- see the side-by-side cell further down.

cohort sizes:  queued=0   boundary=0   measurement=6

=== Simulation sojourn by cohort (truncated at the boundary) ===
  measurement          n=   6  total=   252.47 min  mean=  42.08 min
  measurement_queued   n=   6  total=   252.47 min  mean=  42.08 min
  all                  n=   6  total=   252.47 min  mean=  42.08 min

=== Same, but true end-to-end times (NOT optimizer-comparable) ===
  measurement          n=   6  total=   252.47 min  mean=  42.08 min
  measurement_queued   n=   6  total=   252.47 min  mean=  42.08 min
  all                  n=   6  total=   252.47 min  mean=  42.08 min

mean_sojourn   whole run                    = 30.46 min
mean_sojourn   measured phase only           = 42.08 min
mean_sojourn   measured + queued + boundary  = 42.08 min
queueing_summary(+cohorts): n_finished=6  W=42.08  W_q=11.51  L_sim=2.215


In [15]:
# =============================================================================
# Grid-feasible FIFO -- the like-for-like number to compare an optimum against
# =============================================================================
# The DES runs in continuous time: a connector passes to the next vehicle the
# INSTANT the previous one finishes. The offline models put every plug-in and
# departure on a `delta` slot boundary (assumption A1), so each handover costs
# the successor up to `delta` minutes. The continuous-time means printed above
# are therefore NOT targets any grid schedule can hit -- comparing an offline
# optimum against them directly is apples-to-oranges, and the optimum will
# usually look "worse" for no real reason.
#
# grid_sojourn_by_cohort replays THIS run's own schedule on the grid: group the
# finished vehicles by the lane they actually used (pile AND connector), walk
# each lane's handover chain in service order, and give every vehicle
# ceil(service_minutes / delta) slots, starting no earlier than both its own
# release slot and its predecessor's discretized departure. Only the part of a
# stay inside the measured window counts, so a vehicle already charging at the
# boundary is not charged for its warm-up time -- the same convention the
# offline models use when they place it at a=0.
#
# Round each STAY up once, never its two endpoints independently: doing the
# latter compresses the stay (an 18.21-minute stay squeezed into 18 slots) and
# yields departure times no policy can achieve -- which the offline model
# rejects outright via its earliest-departure bound (24).
grid = m.grid_sojourn_by_cohort(DELTA, since=m.warmup_period)
cont = m.sojourn_by_cohort(since=m.warmup_period)

print(f"=== FIFO replayed on the delta={DELTA:g} min grid ===")
print(f"{'cohort':20s} {'n':>4s} {'cont mean':>10s} {'grid mean':>10s} "
      f"{'grid total':>11s} {'sum D_j':>8s}")
for name in cont:
    g = grid[name]
    print(f"{name:20s} {int(g['n']):4d} {cont[name]['mean_sojourn']:10.2f} "
          f"{g['mean_sojourn']:10.2f} {g['total_sojourn']:11.2f} "
          f"{g['total_departure_slots']:8.0f}")

# `sum D_j` is in the offline objective's own slot units, so it lines up
# directly with ConnectorLaneSolution.objective / DWSolution.UB whenever the
# objective was built over the matching cohorts -- e.g. compare the "all" row
# against a solve with objective_cohorts=COHORTS_ALL.
#
# CAVEATS. Like every per-customer metric here this covers FINISHED vehicles
# only, so with a short measured phase the offline model (which reports every
# vehicle it was given, censoring unfinished ones at D_j = K) is averaging a
# different population -- watch the n column. And this replays OCCUPANCY, not
# power: it assumes each vehicle still needs the same charging time after being
# shifted onto the grid. Shifting changes which vehicles overlap, hence how a
# pile's modules are shared, so treat it as a close reconstruction rather than
# a guaranteed-feasible schedule for the offline model.

=== FIFO replayed on the delta=1 min grid ===
cohort                  n  cont mean  grid mean  grid total  sum D_j
measurement             6      42.08      42.83      257.00      459
measurement_queued      6      42.08      42.83      257.00      459
all                     6      42.08      42.83      257.00      459


In [ ]:
# Offline IP + continuous relaxations on the same station layout and the same
# arrival stream the episode already realized (no synthetic arrivals after
# simulation end). VehicleData uses each EV's arrival, battery, s_i, s_f.
cl_station = StationSpec(
    n_piles=SCENARIO["n_piles"],
    n_connectors=SCENARIO["n_connectors"],
    n_modules=SCENARIO["n_modules"],
    p_module=SCENARIO["p_module"],
)

# Offline MILP / RP controls (same arrivals + station as the finished episode)

MIP_GAP = 1e-3
TIME_LIMIT = 600.0  # seconds per solve; raise if the gap does not close
# Manual completion horizon for the offline models (minutes). This models
# just the MEASURED phase -- OFFLINE_HORIZON = MAX_TIME (not the total
# warm-up + measured duration the DES actually ran for; see engine.max_time
# below). Must be long enough that every arrived EV can finish.
OFFLINE_HORIZON = MAX_TIME
# The DES always runs to its own total horizon (warm-up + measured) exactly
# -- this checks that actually happened (a genuine invariant regardless of
# warm-up), not that it matches OFFLINE_HORIZON (which by design only
# covers the measured phase, not warm-up).
if not np.isclose(env.engine.max_time, float(env.engine.current_time)):
    raise ValueError("Engine's total horizon and actual finish time are not equal")


Offline instance: J=6 vehicles (arrived during the measured phase only, warmup_period=360.0 min), horizon T=120.0 min, delta=1 min, 


In [17]:
# =============================================================================
# Building the measured-window offline instance: the boundary flags
# =============================================================================
# build_measurement_instance turns a finished simulation into a ready-to-solve
# instance in one call: it shifts arrivals back to t=0, optionally folds in
# the boundary queue and the already-charging vehicles, and tags every vehicle
# with its Cohort. Two independent flags:
#
#   include_queued : bool
#       True  -> vehicles waiting in the queue at the boundary are added as
#                ordinary vehicles with a=0 ("already here"). Their SoC has
#                not moved since they arrived, so nothing else changes.
#       False -> the measured phase is modelled as starting with an empty
#                queue (matches FLUSH_QUEUE_AT_WARMUP=True, or simply
#                choosing to ignore the warm-up backlog).
#
#   boundary_mode : BoundaryMode | None    (vehicles already PLUGGED IN)
#       None              -> leave them out entirely; every connector starts
#                            free. The most optimistic assumption.
#       BoundaryMode.FIXED    -> the optimizer has NO control over them.
#                            Their occupancy and power are pinned to exactly
#                            what the simulation did (reconstructed exactly,
#                            see below) and their departure is floored to the
#                            slot grid. They still hold connectors and draw
#                            modules -- that is the whole point -- but their
#                            D_j is a constant, so normally exclude them from
#                            the objective (next cell).
#       BoundaryMode.OPTIMIZE -> the optimizer DOES control their future power
#                            and departure, but never their pile/connector
#                            (unplugging a mid-charge car is unrealistic).
#                            Energy accounting starts from what they already
#                            have, so they only need W_j - initial_energy more.
#
# How the FIXED/seed power is reconstructed: the simulation's charge_trace is
# event-sampled, so each slot's power is computed as the EXACT energy
# delivered in that slot divided by the slot length (the energy-equivalent
# constant power, same idea as tau^delta). Holding each sample's p_act across
# the slot instead would both overstate energy and combine powers from
# allocation regimes that never coexisted -- which can push a pile over its
# module limit and make the model infeasible.
from offline_cl_opt.boundary import (
    BoundaryMode,
    Cohort,
    build_measurement_instance,
    COHORTS_MEASUREMENT,          # objective over measured-phase arrivals only
    COHORTS_MEASUREMENT_QUEUED,   # ... plus those queued at the boundary
    COHORTS_ALL,                  # ... plus those already plugged in
)

BOUNDARY_MODE = BoundaryMode.OPTIMIZE   # try FIXED, or None
INCLUDE_QUEUED = True                   # try False

inst = build_measurement_instance(
    arrived_post_warmup=m.arrived_post_warmup,
    queued_at_warmup_end=m.queued_at_warmup_end,
    in_service_at_warmup_end=m.in_service_at_warmup_end,
    warmup_period=env.engine.warmup_period,
    delta=DELTA,
    horizon_minutes=float(OFFLINE_HORIZON),
    include_queued=INCLUDE_QUEUED,
    boundary_mode=BOUNDARY_MODE,
    # Passing the station runs the whole-module check (20) on FIXED vehicles
    # here, where it can name the offending pile-slot, instead of letting it
    # surface later as a bare INFEASIBLE.
    station=cl_station,
)

from collections import Counter
print(f"instance: J={len(inst.vehicles)} vehicles, "
      f"cohorts={dict(Counter(c.value for c in inst.cohorts.values()))}")
print("boundary vehicles:", {j: (bv.mode.value, bv.pile) for j, bv in inst.boundary_vehicles.items()})
# Vehicles dropped because their floored departure landed at slot 0 -- they
# left within the first delta of the window, so they occupy no whole slot.
# They are removed from the vehicle list, the boundary map AND the cohort
# tags together, so nothing downstream can mistake one for a fresh arrival.
print("discarded (departure_slot == 0):", inst.discarded)

instance: J=6 vehicles, cohorts={'measurement': 6}
boundary vehicles: {}
discarded (departure_slot == 0): []


In [19]:
# =============================================================================
# Which vehicles does the OBJECTIVE minimise over?  (objective_cohorts)
# =============================================================================
# Separate from "who is in the model" is "who does the optimizer work for".
# A vehicle outside objective_cohorts is still modelled in full -- it holds
# its connector, draws its modules, constrains everyone else -- its D_j just
# carries no weight in the sum being minimised.
#
#   COHORTS_MEASUREMENT        -- only vehicles that arrived in the window
#   COHORTS_MEASUREMENT_QUEUED -- plus those queued at the boundary
#   COHORTS_ALL                -- plus those already plugged in
#
# Excluding BOUNDARY is the natural choice with BoundaryMode.FIXED: their
# D_j is pinned, so including it only adds a constant to the objective and
# dilutes the reported mean with a number the optimizer never chose.
#
# With BoundaryMode.OPTIMIZE, though, watch out: excluding them means the
# optimizer has no incentive to finish them, so it will happily let them
# linger. Their sojourn in the "all" report will jump accordingly. If you
# want them served promptly, put them IN the objective.
#
# Whatever you optimise, the solution reports total/mean sojourn for ALL
# THREE nested cohorts, so you can always see the other two.
from offline_cl_opt import build_cl_model, solve_cl_model
from offline_cl_opt.solution import extract_solution as extract_cl

# for label, sel in [("measurement only", COHORTS_MEASUREMENT),
#                    ("measurement+queued", COHORTS_MEASUREMENT_QUEUED),
#                    ("all three cohorts", COHORTS_ALL)]:
#for label, sel in [("all three cohorts", COHORTS_ALL)]:
for label, sel in [("measurement only", COHORTS_MEASUREMENT)]:
    cl = build_cl_model(
        inst.vehicles, cl_station,
        boundary_vehicles=inst.boundary_vehicles,   # boundary conditions
        cohorts=inst.cohorts,                       # who is in which cohort
        objective_cohorts=sel,                      # who the objective serves
        # NOTE break_symmetry must stay False when boundary vehicles are
        # present: their lane is pinned to wherever they physically are,
        # which has no reason to respect the canonical ordering (25)-(26).
        break_symmetry=True,
        bound_departures=True,
        tie_break=False,
        delta=DELTA,
        horizon_minutes=OFFLINE_HORIZON,
    )
    
    solve_cl_model(cl, mip_gap=1e-3, time_limit=None, verbose=True, cutoff=None)

    sol = extract_cl(cl)
    print(f"\nobjective_cohorts = {label}")
    print(f"  objective={sol.objective:.1f}  optimised over {sol.n_optimized}"
          f"/{sol.n_vehicles} vehicles  mean={sol.mean_sojourn:.2f} min")
    for name, s in sol.by_cohort.items():
        print(f"    {name:20s} n={int(s['n']):4d}  total={s['total_sojourn']:9.2f}"
              f"  mean={s['mean_sojourn']:7.2f} min")

Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.001
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0.001

Academic license 2616613 - for non-commercial use only - registered to m.___@mail.utoronto.ca
Optimize a model with 4974 rows, 2339 columns and 29937 nonzeros (Min)
Model fingerprint: 0xa004927b
Model has 1036 linear objective coefficients and an objective constant of 720
Variable types: 1554 continuous, 785 integer (545 binary)
Coefficient statistics:
  Matrix range     [2e-02, 2e+02]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 4e+02]

Found heuristic solution: objective 720.0000000
Presolve removed 949 rows and 506 columns
Presolve time: 0.07s
Presolved: 4025 rows, 1833 

In [ ]:
# The per-vehicle table now carries cohort / boundary_mode / in_objective, plus
# energy_delivered_before_kwh so a boundary vehicle's in-window energy does not
# read as a shortfall against its full requirement.
display(sol.per_vehicle.head(12))

In [26]:
len(inst.vehicles), inst.cohorts

(6,
 {11: <Cohort.MEASUREMENT: 'measurement'>,
  12: <Cohort.MEASUREMENT: 'measurement'>,
  13: <Cohort.MEASUREMENT: 'measurement'>,
  14: <Cohort.MEASUREMENT: 'measurement'>,
  15: <Cohort.MEASUREMENT: 'measurement'>,
  16: <Cohort.MEASUREMENT: 'measurement'>})

In [ ]:
# =============================================================================
# Same flags on the Dantzig-Wolfe decomposition
# =============================================================================
# solve_by_decomposition takes exactly the same three arguments, with the same
# meaning, and reports the same by_cohort breakdown. LB/UB (and the sojourn
# figures derived from them) always cover the objective's own cohorts -- there
# is no per-cohort LB, since the bound is one scalar certifying the objective
# as a whole.
from offline_cl_dw import solve_by_decomposition

# mean-sojourn uncertainty (minutes) = δ · g / J
# gap = mean-soj uncertainty * J / δ
# To target an m-minute mean-sojourn tolerance:
# gap_tolerance = m * len(vehicles) / delta

# objective_cohorts = COHORTS_MEASUREMENT, COHORTS_MEASUREMENT_QUEUED, COHORTS_ALL
dw_sol, dw_cg = solve_by_decomposition(
    inst.vehicles, cl_station,
    boundary_vehicles=inst.boundary_vehicles,
    cohorts=inst.cohorts,
    objective_cohorts=COHORTS_ALL,   # same three choices as above
    delta=DELTA,
    horizon_minutes=OFFLINE_HORIZON,
    gamma=0.0,
    time_limit=None,
    purge_every=None,
    #gap_tolerance=.000001,
    gap_tolerance=len(inst.vehicles) * 1 / DELTA
    progress=True,
)
print(f"DW bracket: LB={dw_sol.LB:.2f}  UB={dw_sol.UB:.2f}  gap={dw_sol.gap:.2f}"
      f"  converged={dw_sol.converged}  optimised over {dw_sol.n_optimized}/{dw_sol.n_vehicles}")
print(f"  mean sojourn  LB={dw_sol.mean_sojourn_LB:.2f}  UB={dw_sol.mean_sojourn_UB:.2f} min")
for name, s in dw_sol.by_cohort.items():
    print(f"    {name:20s} n={int(s['n']):4d}  mean={s['mean_sojourn']:7.2f} min")

# TIP: boundary vehicles get no "never served" fallback column, so their
# seeded column is forced into the capacity rows. The seed used is the
# simulation's own realized trajectory, which is jointly feasible by
# construction. If price-and-branch's UB comes out loose, feed it more
# columns from the simulation itself:
#     from offline_cl_dw import columns_from_evs
#     extra = columns_from_evs(m.finished_evs, {v.id: v for v in inst.vehicles},
#                              DELTA, dw_sol.K)
#     solve_by_decomposition(..., extra_seed_columns=extra)

[colgen] iter 1: z_RMP=552.990, best_LB=189.085, gap=363.905, exact=True, candidates=6, columns_added=6, total_columns=18
[colgen] iter 2: z_RMP=549.551, best_LB=189.946, gap=359.605, exact=True, candidates=6, columns_added=6, total_columns=24
[colgen] iter 3: z_RMP=545.891, best_LB=237.423, gap=308.468, exact=True, candidates=6, columns_added=6, total_columns=30
[colgen] iter 4: z_RMP=542.006, best_LB=244.587, gap=297.419, exact=True, candidates=6, columns_added=6, total_columns=36
[colgen] iter 5: z_RMP=538.184, best_LB=245.935, gap=292.250, exact=True, candidates=4, columns_added=4, total_columns=40
[colgen] iter 6: z_RMP=529.910, best_LB=245.935, gap=283.975, exact=True, candidates=4, columns_added=4, total_columns=44
[colgen] iter 7: z_RMP=524.443, best_LB=245.935, gap=278.508, exact=True, candidates=4, columns_added=4, total_columns=48
[colgen] iter 8: z_RMP=520.955, best_LB=254.186, gap=266.769, exact=True, candidates=4, columns_added=4, total_columns=52
[colgen] iter 9: z_RMP=5

In [44]:
# =============================================================================
# Apples-to-apples: simulation vs. optimizer on the SAME cohort
# =============================================================================
# Two things must match for the comparison to mean anything:
#   1. the same cohort on both sides (use the same level name), and
#   2. the same clock -- the simulation must truncate the boundary cohorts at
#      the boundary (truncate_included=True), because the optimizer places
#      those vehicles at a=0 and so never sees their pre-boundary time.
#
# Four columns:
#   sim    -- continuous-time FIFO (metrics.sojourn_by_cohort, truncated)
#   grid   -- FIFO's OWN schedule replayed on the offline model's delta grid
#             (metrics.grid_sojourn_by_cohort) -- the like-for-like number,
#             since the model can only depart on slot boundaries (assumption
#             A1); "sim" is not a target any grid schedule can actually hit.
#   opt    -- the exact model's UB (sol.by_cohort)
#   dw     -- NOTE: dw_sol.mean_sojourn_LB is a SINGLE scalar certifying the
#             whole objective dw_sol was solved over (objective_cohorts), not
#             a per-cohort bound -- see DWSolution's own docstring, "There is
#             no per-cohort LB". The same number is repeated on every row
#             below; it is only a valid LB for whichever cohort was actually
#             passed as objective_cohorts when dw_sol was solved, not for the
#             other two rows. Re-solve with objective_cohorts set to a single
#             cohort if you need a real bound for that cohort specifically.
sim_by_cohort = m.sojourn_by_cohort(since=m.warmup_period)        # truncated
grid_by_cohort = m.grid_sojourn_by_cohort(DELTA, since=m.warmup_period)

print(f"{'cohort':22s} {'sim mean':>9s} {'grid mean':>10s} {'opt mean':>8s} {'dw LB':>9s} "
      f"{'sim n':>6s} {'grid n':>7s} {'opt n':>6s} {'dw n':>6s}")
for name in ("measurement", "measurement_queued", "all"):
    s, g, o, d = sim_by_cohort[name], grid_by_cohort[name], sol.by_cohort[name], dw_sol
    print(f"{name:22s} {s['mean_sojourn']:9.2f} {g['mean_sojourn']:10.2f} "
          f"{o['mean_sojourn']:9.2f} {d.mean_sojourn_LB:9.2f} "
          f"{int(s['n']):6d} {int(g['n']):7d} {int(o['n']):6d} {int(d.by_cohort[name]['n']):6d}")

# CAVEAT (read the n columns): the simulation averages only vehicles that
# FINISHED inside the run, while the offline model reports every vehicle it
# was given, censoring any that are unfinished at the horizon at D_j = K
# (see connector_lane_model.html, Section 6.3). When those counts differ, the
# means are not measuring the same population -- lengthen MAX_TIME until
# essentially everything finishes before reading the gap as "how much the
# policy leaves on the table".


cohort                  sim mean  grid mean opt mean     dw LB  sim n  grid n  opt n   dw n
measurement                42.08      42.83     41.83     37.77      6       6      6      6
measurement_queued         42.08      42.83     41.83     37.77      6       6      6      6
all                        42.08      42.83     41.83     37.77      6       6      6      6


In [ ]:

# Vehicles for the offline model: only arrived_post_warmup (not the whole
# run's arrived_evs, and not the raw oversampled `evs` draw) -- since
# OFFLINE_HORIZON only spans the MEASURED phase, feeding it a warm-up-era
# arrival would be wrong twice over: it wasn't part of the window being
# modeled, and (see the arrival_time shift below) its clock value predates
# what "t=0" means for this offline instance anyway.
#
# arrival_time on every EV is still stamped in *whole-run* clock (e.g. 2450
# if warmup_period=2400), but the offline model's own horizon starts at
# t=0 for the measured window -- shift every vehicle's `a` back by
# warmup_period so k_j = ceil(a_j/delta) lands inside [0, OFFLINE_HORIZON)
# instead of raising "release slot falls outside the horizon".
#
# NOTE this cell builds the MEASUREMENT-ONLY instance: it does not
# include queued_at_warmup_end / in_service_at_warmup_end. That is one
# of the three valid choices, not a limitation -- the cells below build
# the boundary-inclusive instances with build_measurement_instance and
# show how to pick which cohorts the objective serves.
import dataclasses
vehicles = [
    dataclasses.replace(v, a=v.a - env.engine.warmup_period)
    for v in vehicles_from_evs(env.engine.metrics.arrived_post_warmup)
]
print(
    f"Offline instance: J={len(vehicles)} vehicles "
    f"(arrived during the measured phase only, warmup_period={env.engine.warmup_period:.1f} min), "
    f"horizon T={OFFLINE_HORIZON:.1f} min, delta={DELTA:g} min, "
)

In [ ]:

from offline_cl_dw import solve_by_decomposition
# mean-sojourn uncertainty (minutes) = δ · g / J
# gap = mean-soj uncertainty * J / δ
# To target an m-minute mean-sojourn tolerance:
# gap_tolerance = m * len(vehicles) / delta

dw_solution, dw_colgen = solve_by_decomposition(
    vehicles, cl_station, delta=DELTA, horizon_minutes=OFFLINE_HORIZON,
    progress=True,
    gamma=0.0,
    time_limit=None,
    purge_every=None,
    gap_tolerance=.000001,
)

print(f"\nbracket: [{dw_solution.LB:.3f}, {dw_solution.UB:.3f}]  "
      f"gap={dw_solution.gap:.3f}  converged={dw_solution.converged}  "
      f"iterations={dw_solution.iterations}")
# rmp_gap is the colgen stopping-criterion gap (z_RMP - LB) -- a diagnostic
# of how close column generation got to proving z_RMP == z_MP, NOT itself a
# certified bound the way `gap` (UB - LB) is.
print(f"rmp_gap (z_RMP - LB, not certified): {dw_solution.rmp_gap:.3f}")
#print(f"compact-model exact objective (from 'Build and solve' above) = {solution.objective:.3f}")
print(f"whole_module_feasible: {dw_solution.whole_module_feasible}")
#dw_solution.per_vehicle
print(
    f"Total sojourn bracket: "
    f"[{dw_solution.total_sojourn_LB:.2f}, "
    f"{dw_solution.per_vehicle['sojourn_min'].sum():.2f}]"
)

In [35]:
print(f"Sim total sojourn (finished): {sim_total_sojourn:.2f} min")
print(f"Sim mean sojourn (finished):  {sim_mean_sojourn:.2f} min")
dw_solution.mean_sojourn_LB

Sim total sojourn (finished): 541.80 min
Sim mean sojourn (finished):  41.68 min


30.442782458044313

In [29]:
print(f"Sim total sojourn (finished): {sim_total_sojourn:.2f} min")
print(f"Sim mean sojourn (finished):  {sim_mean_sojourn:.2f} min")
dw_solution.mean_sojourn_LB

Sim total sojourn (finished): 236.50 min
Sim mean sojourn (finished):  47.30 min


40.4628576492847

In [42]:

cl_model = build_cl_model(vehicles, cl_station, break_symmetry=True, bound_departures=True,
delta=DELTA, horizon_minutes=OFFLINE_HORIZON, tie_break=False)
solve_cl_model(cl_model, mip_gap=1e-3, time_limit=None, verbose=True, cutoff=None)
solution = extract_solution(cl_model)

print(f"status={solution.status}, gap={solution.mip_gap:.2%}, runtime={solution.runtime:.2f}s")
print(f"objective (sum D_j) = {solution.objective:.3f} slots, mean sojourn = {solution.mean_sojourn:.3f} min")
solution.per_vehicle

Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.001
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0.001

Academic license 2616613 - for non-commercial use only - registered to m.___@mail.utoronto.ca
Optimize a model with 11335 rows, 3733 columns and 264444 nonzeros (Min)
Model fingerprint: 0x5b8a1b6f
Model has 1532 linear objective coefficients and an objective constant of 1560
Variable types: 2298 continuous, 1435 integer (1195 binary)
Coefficient statistics:
  Matrix range     [3e-02, 2e+02]
  Objective range  [1e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 3e+02]

Found heuristic solution: objective 1560.0000000
Presolve removed 1634 rows and 525 columns
Presolve time: 1.00s
Presolved: 9701 rows

,vehicle_id,arrival,served,pile,connector,start_slot,departure_slot,sojourn_min,energy_kwh,energy_required_kwh
0,0,2.0,True,0.0,0.0,1.0,13.0,24.0,35.926494,35.926494
1,1,6.0,True,0.0,1.0,3.0,16.0,26.0,29.388572,29.388572
2,2,8.0,False,NaN,NaN,60.0,60.0,112.0,0.000000,38.487826
3,3,10.0,True,1.0,0.0,5.0,14.0,18.0,34.438342,34.438342
4,4,14.0,True,1.0,1.0,7.0,24.0,34.0,42.910430,42.910430
5,5,26.0,True,0.0,0.0,13.0,25.0,24.0,37.551944,37.551944
6,6,28.0,True,1.0,1.0,26.0,41.0,54.0,35.521619,35.521619
7,7,32.0,True,0.0,1.0,17.0,32.0,32.0,40.036083,40.036083
8,8,44.0,True,1.0,0.0,22.0,35.0,26.0,38.527586,38.527586
9,9,46.0,False,NaN,NaN,60.0,60.0,74.0,0.000000,47.784760


In [12]:

cl_model = build_cl_model(vehicles, cl_station, break_symmetry=True, bound_departures=True,
delta=DELTA, horizon_minutes=OFFLINE_HORIZON, tie_break=False)
solve_cl_model(cl_model, mip_gap=5e-3, time_limit=None, verbose=True, cutoff=None)
solution = extract_solution(cl_model)

print(f"status={solution.status}, gap={solution.mip_gap:.2%}, runtime={solution.runtime:.2f}s")
print(f"objective (sum D_j) = {solution.objective:.3f} slots, mean sojourn = {solution.mean_sojourn:.3f} min")
solution.per_vehicle

Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.005
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0.005

Academic license 2616613 - for non-commercial use only - registered to m.___@mail.utoronto.ca
Optimize a model with 5520 rows, 2572 columns and 40040 nonzeros (Min)
Model fingerprint: 0x5de6fd04
Model has 1144 linear objective coefficients and an objective constant of 960
Variable types: 1716 continuous, 856 integer (616 binary)
Coefficient statistics:
  Matrix range     [2e-02, 2e+02]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 4e+02]

Found heuristic solution: objective 960.0000000
Presolve removed 1131 rows and 559 columns
Presolve time: 0.09s
Presolved: 4389 rows, 2013

,vehicle_id,arrival,served,pile,connector,start_slot,departure_slot,sojourn_min,energy_kwh,energy_required_kwh
0,0,3.438384,True,0.0,0.0,4.0,29.0,25.561616,39.681788,39.681788
1,1,17.464539,True,0.0,1.0,18.0,55.0,37.535461,51.053306,51.053306
2,2,24.296597,True,0.0,0.0,33.0,57.0,32.703403,28.869372,28.869372
3,3,31.411370,False,NaN,NaN,120.0,120.0,88.588630,0.000000,53.854608
4,4,44.757326,True,0.0,1.0,55.0,76.0,31.242674,34.778222,34.778222
5,5,78.553507,True,0.0,1.0,79.0,98.0,19.446493,34.124377,34.124377
6,6,86.074832,True,0.0,0.0,87.0,117.0,30.925167,38.012206,38.012206
7,7,97.833511,False,NaN,NaN,120.0,120.0,22.166489,0.000000,35.817405


In [11]:
solution.total_sojourn

219.33986859282697

In [ ]:
from offline_cl_opt import solve_cl_model_adaptive

result = solve_cl_model_adaptive(vehicles, cl_station, delta=DELTA, break_symmetry=True,
                        progress=True, verbose=False,
                        bound_departures=True, cutoff=None,
                        time_limit=None,
                        horizon_minutes=OFFLINE_HORIZON, mip_gap=5e-3,
                        warm_start_from_conservative=False)
adaptive_solution = extract_solution(result.cl_model)

print(f"converged={result.converged}, iterations={result.iterations}")
print(f"objective history: {result.objective_history}")
print(f"pile-slots promoted to integer: {len(result.integer_pile_slots)}")
print(f"adaptive objective = {adaptive_solution.objective:.3f} slots, mean sojourn = {adaptive_solution.mean_sojourn:.3f} min")
adaptive_solution.per_vehicle